# Setup

- presupposes call to extract_Erates_and_Pindex_from_MFMOD.py to create the dataframe used below

In [ ]:
# Load data, extract exchange rates, and normalize them
import pandas as pd

# Load the CSV file into a pandas DataFrame
file_path = 'priceIndex_and_exchangeRate_series_2025_SP.csv'  # Replace with your CSV file path
df = pd.read_csv(file_path, index_col=0)

# Filter out data before 2000 and non-African countries
ma_window = 5 #years looking back in moving average
start_date = pd.to_datetime("2000")
sd_minus_4 = start_date - pd.DateOffset(years=4) 
df.index = pd.to_datetime(df.index)
df = df[(df.index >= sd_minus_4) & (df.index <= '2023-01-01')]
fX = pd.DataFrame(index=df.index)

african_countries = [
    'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COG', 'CIV', 
    'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN', 
    'LBY', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 
    'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA', 
    'ZMB', 'ZWE'
]

CFA_countries = [
    'CMR', 'CAF', 'TCD', 'GNQ', 'GAB', 'COG', 'BEN', 'BFA', 'CIV', 'GNB', 'MLI', 'NER', 'SEN', 'TGO'
]

commodity_exporters = ['AGO', 'DZA', 'NGA', 'ZAF', 'EGY', 'MAR', 'TUN', 'CIV', 'GHA', 'BWA', 'NAM']
commodity_exportert = ['NGA', 'AGO', 'DZA', 'LBY', 'COG', 'GAB', 'TCD', 'SSD', 'GNQ', 'COD', 'ZMB', 'GIN']


# Create a mapping from ISO3 code to full country name
iso3_to_country_name = {
    'DZA': 'Algeria', 'AGO': 'Angola', 'BEN': 'Benin', 'BWA': 'Botswana', 'BFA': 'Burkina Faso',
    'BDI': 'Burundi', 'CMR': 'Cameroon', 'CPV': 'Cape Verde', 'CAF': 'Central African Republic',
    'TCD': 'Chad', 'COM': 'Comoros', 'COG': 'Congo', 'CIV': 'Ivory Coast', 'COD': 'Democratic Republic of the Congo',
    'DJI': 'Djibouti', 'EGY': 'Egypt', 'GNQ': 'Equatorial Guinea', 'ERI': 'Eritrea', 'SWZ': 'Eswatini',
    'ETH': 'Ethiopia', 'GAB': 'Gabon', 'GMB': 'Gambia', 'GHA': 'Ghana', 'GIN': 'Guinea', 'GNB': 'Guinea-Bissau',
    'KEN': 'Kenya', 'LSO': 'Lesotho', 'LBR': 'Liberia', 'MDG': 'Madagascar', 'MWI': 'Malawi', 'MLI': 'Mali',
    'MRT': 'Mauritania', 'MUS': 'Mauritius', 'MAR': 'Morocco', 'MOZ': 'Mozambique', 'NAM': 'Namibia',
    'NER': 'Niger', 'NGA': 'Nigeria', 'RWA': 'Rwanda', 'STP': 'Sao Tome and Principe', 'SEN': 'Senegal',
    'SYC': 'Seychelles', 'SLE': 'Sierra Leone', 'SOM': 'Somalia', 'ZAF': 'South Africa', 'SSD': 'South Sudan',
    'SDN': 'Sudan', 'TZA': 'Tanzania', 'TGO': 'Togo', 'TUN': 'Tunisia', 'UGA': 'Uganda', 'ZMB': 'Zambia',
    'ZWE': 'Zimbabwe', 'LBY': 'Libya'	
}


# Create a dictionary to store the new columns
exchange_rates = {}
first_non_nan_indices = {}
# Populate the dictionary with the new columns
for country in african_countries:
    panusatls_col = f"{country}PANUSATLS"
    price_level_col = f"{country}NECONPRVTXN"
    usa_price_level_col = "USANECONPRVTXN"
    
    if panusatls_col in df.columns and price_level_col in df.columns:
        exchange_rates[f"{country}_RER"] = (df[panusatls_col] * df[usa_price_level_col] / df[price_level_col])
        exchange_rates[f"{country}_NER"] = df[panusatls_col]
        exchange_rates[f"{country}_R2NER"] = (df[panusatls_col] / df[price_level_col])
        first_idx = exchange_rates[f"{country}_RER"].first_valid_index()
        first_non_nan_indices[country] = first_idx
    else:
        print(f"Missing columns for {country}: {panusatls_col}, {price_level_col}")

# Add # Find the first non-NaN index for each country's _RER column
#all the new columns to fX at once
fX = pd.concat([fX, pd.DataFrame(exchange_rates)], axis=1)
fX["USANECONPRVTXN"] = df["USANECONPRVTXN"]


normalised_rates = {}

for country in african_countries:
    rer_col = f"{country}_RER"
    ner_col = f"{country}_NER"
    r2ner_col = f"{country}_R2NER"
    
    if rer_col in fX.columns and ner_col in fX.columns and r2ner_col in fX.columns and first_non_nan_indices[country] is not None:
        # Normalize the column and add moving average to r2ner    
        normalised_rates[f"{rer_col}_normalized"] = fX[rer_col] / fX[rer_col].loc[first_non_nan_indices[country]] * 100
        normalised_rates[f"{ner_col}_normalized"] = fX[ner_col] / fX[ner_col].loc[first_non_nan_indices[country]] * 100
        normalised_rates[f"{r2ner_col}_normalized"] = fX[r2ner_col] / fX[r2ner_col].loc[first_non_nan_indices[country]] * 100
        normalised_rates[f"US_price_level_normalized_to_{country}_base"] = fX["USANECONPRVTXN"] / fX["USANECONPRVTXN"].loc[first_non_nan_indices[country]] * 100
        normalised_rates[f"{country}_R2NER_MA"] = normalised_rates[f"{r2ner_col}_normalized"].rolling(window=ma_window).mean()
    else:
        print(f"Missing data for {country}: {rer_col} or {ner_col}")
# Add all the new columns to fX at once
fX = pd.concat([fX, pd.DataFrame(normalised_rates)], axis=1)

import numpy as np

def insert_rise(x, y, idx, y0=100.0):
    """
    Insert a point at the crossing with y0.
    """
    t = (y0 - y[idx]) / (y[idx + 1] - y[idx])  # 0 < t < 1
    x_cross = x[idx] + t * (x[idx + 1] - x[idx])
    y_cross = y0  # exactly on the threshold

    x_aug = np.insert(x, idx + 1, x_cross)
    y_aug = np.insert(y, idx + 1, y_cross)

    return pd.to_datetime(x_aug), np.asarray(y_aug)


def insert_return(x, y, idx, y0=100.0):
    """
    Insert a point at the crossing with y0.
    """
    t = (y0 - y[idx]) / (y[idx] - y[idx-1])  # 0 < t < 1
    x_cross = x[idx] + t * (x[idx] - x[idx-1])
    y_cross = y0  # exactly on the threshold

    x_aug = np.insert(x, idx, x_cross)
    y_aug = np.insert(y, idx, y_cross)

    return pd.to_datetime(x_aug), np.asarray(y_aug)

def extend_series(x, y, change_idx, y0=100.0):

    x_aug = []
    y_aug = []
    last = 0
    for i in change_idx:
        # original samples up to i
        x_aug.extend(x[last:i + 1])
        y_aug.extend(y[last:i + 1])

        # linear interpolation of the crossing with y0
        t = (y0 - y[i]) / (y[i + 1] - y[i])        # 0 < t < 1
        x_cross = x[i] + t * (x[i + 1] - x[i])

        x_aug.append(x_cross)
        y_aug.append(y0)                            # exactly on the threshold
        last = i + 1

    # append the remaining tail
    x_aug.extend(x[last:])
    y_aug.extend(y[last:])

    x_aug = pd.to_datetime(x_aug)
    y_aug = np.asarray(y_aug)

    # integrate ONLY the part above the line
    return x_aug, y_aug


In [ ]:
# indentify the material depreciation events in real to nominal LCU:USD exchange rate
import numpy as np
from scipy.signal import argrelextrema

# Define the thresholds for identifying peaks and troughs
rise_threshold = 0.15  # 20% increase threshold
return_threshold = 0  # 15% return threshold


results = []
results_extended = {}
for country in african_countries:
    r2ner_col = f"{country}_R2NER_normalized"
    ma_col = f"{country}_R2NER_MA"
    if r2ner_col not in fX.columns:
        continue
    s = fX[r2ner_col]#.dropna()
    if s.empty:
        continue
    values = s[s.index >= start_date].values
    times = s[s.index >= start_date].index
    ma_5year = fX.loc[fX.index >= start_date, ma_col].values


    trough_indices = argrelextrema(ma_5year, np.less)[0]
    first_non_nan_index = np.where(~np.isnan(ma_5year))[0][0]
    if ma_5year[first_non_nan_index] < ma_5year[first_non_nan_index+1]:
        trough_indices = np.insert(trough_indices, 0, first_non_nan_index)
    peak_indices = argrelextrema(values, np.greater)[0]
    if values[-1] > values[-2]:
        peak_indices = np.append(peak_indices, len(values) - 1)

    for t_idx in trough_indices:
        possible_peaks = peak_indices[peak_indices > t_idx]
        if possible_peaks.size == 0:
            continue
        # p_idx = possible_peaks[0] # maybe loop over all peaks? or pick the highest peak?
        increase = values[p_idx] / ma_5year[t_idx] - 1.0
        if increase >= rise_threshold:
            threshold = ma_5year[t_idx] * (1. + return_threshold)
            fall_date = None
            fall_period = None
            area_under_curve = None
            gain_at_fall = None

            # Use for-else construct: if no break occurs, use the last value.
            for idx in range(p_idx + 1, len(values)):
                if values[idx] < threshold:
                    fall_date = times[idx]
                    return_period = times[idx] - times[t_idx]
                    gain_at_fall = (values[idx] - values[t_idx]) / values[t_idx] * 100
                    values_integrand = np.array(values, copy = True)
                    values_integrand[t_idx], values_integrand[idx] = ma_5year[t_idx], ma_5year[t_idx] # R2NER replaced with baseline at start and end date for clean integration
                    integrand = np.maximum(values_integrand/ma_5year[t_idx] - 1, 0.0)*100 # percentage deviation from baseline at event start date is the integrand
                    integration_period = (times[t_idx:idx+1]-times[t_idx]).days.astype('int64')/365
                    area_under_curve = np.trapezoid(integrand[t_idx:idx+1], integration_period) 
                    break
            else:
                # If fall condition never met, use the last time point
                idx = len(values) - 1
                fall_date = times[idx]
                return_period = times[idx] - times[t_idx]
                gain_at_fall = (values[idx] - values[t_idx]) / values[t_idx] * 100
                values_integrand = np.array(values, copy = True)
                values_integrand[t_idx], values_integrand[idx] = ma_5year[t_idx], ma_5year[t_idx]
                integrand = np.maximum(values_integrand/ma_5year[t_idx] - 1, 0.0)*100
                integration_period = (times[t_idx:idx+1]-times[t_idx]).days.astype('int64')/365
                area_under_curve = np.trapezoid(integrand[t_idx:idx+1], integration_period) 

            results.append({
                "Country": country,
                "Trough_Date": times[t_idx],
                "Peak_Date": times[p_idx],
                "Fall_Date": fall_date,
                "Area_Under_Curve": area_under_curve,
                "Percentage_Increase_to_Peak": increase * 100,
                "Percentage_Increase_at_Return": gain_at_fall,
                "Years_to_return":return_period.days/365 if return_period is not None else None,
                "Trough_Value": values[t_idx],
                "MA_at_Trough": ma_5year[t_idx]})
                
            if country not in results_extended:
                results_extended[country] = []
            results_extended[country].append({
            "values_integrand": values_integrand,
            "times": times,
            })


# df_transitions = remove_nested_transitions(pd.DataFrame(results))            
df_transitions = pd.DataFrame(results)

In [ ]:
df_transitions


In [ ]:
import altair as alt
threshold = ma_5year[t_idx]

df = pd.DataFrame({
    "x": times,
    "y": values,
    "ma": ma_5year,
    "values_integrand": values_integrand,
    "threshold": threshold
})

df_area = pd.DataFrame({
    "x": times[t_idx:idx+1],
    "values_integrand": values_integrand[t_idx:idx+1],
    "threshold": threshold,
})

# --------------------------------------------------------------
# 2.  Build the Altair layers
# --------------------------------------------------------------

base = alt.Chart(df).mark_line(color="#2ca02c").encode(
    x=alt.X("x:T", title="time"),
    y=alt.Y("y:Q", title="R2NER"),
    # y2=alt.Y2("ma:Q", title="5-year moving average"),
)

ma_line = base.mark_line(color="#1f77b4", strokeDash=[5, 5]).encode(
    y=alt.Y("ma:Q", title="5-year moving average")
)

integrand_line = base.mark_line(color="#FF7F0E").encode(
    y=alt.Y("values_integrand:Q", title="Values Integrand (R2NER - 5-year MA)"),
)

threshold_line = base.mark_line(color="#FF0000", strokeDash=[5, 5]).encode(
    y=alt.Y("threshold:Q", title="Threshold (5-year MA * 1.15)"),
)

# Shaded band between the curve and the threshold
area = alt.Chart(df_area).mark_area(opacity=0.25, color="#4682B4").encode(
    x=alt.X("x:T", title="time"),
    y="values_integrand:Q",
    y2=alt.Y2("threshold:Q"))

chart = (base + ma_line + integrand_line + threshold_line + area).properties(
    width=750, height=400,
    title="Integrated area above the threshold"
).interactive()

# chart
points = base.mark_point(filled=True, color="black", size=50, opacity=0).encode(
    tooltip=[
        alt.Tooltip("x:T", title="Date"),
        alt.Tooltip("y:Q", title="R2NER"),
        alt.Tooltip("ma:Q", title="5-year Moving Average"),
        alt.Tooltip("values_integrand:Q", title="Values Integrand")
    ]
)

chart = chart + points
chart


In [ ]:
# summarise the event statistics

import altair as alt

# Ensure Altair can handle large datasets
alt.data_transformers.disable_max_rows()

exclusion_list = [] # ["BFA", "CMR", "TCD", "DJI", "LSO", "SDN", "SWE"]
# df_transitions = pd.DataFrame(results)
df_transitions_to_chart = df_transitions[~df_transitions["Country"].isin(exclusion_list)]
# Histogram of length of drawdown period
chart_days = alt.Chart(df_transitions_to_chart).mark_bar().encode(
    alt.X("Years_to_return:Q", bin=alt.Bin(maxbins=30), title="Years"),
    alt.Y("count()", title="Count")
).properties(
    title="Time to Return",
    width=400,
    height=300
)

# Histogram of Drawdown Area
chart_area = alt.Chart(df_transitions_to_chart).mark_bar().encode(
    alt.X("Area_Under_Curve:Q", bin=alt.Bin(maxbins=30), title="Percentage-Years over Baseline"),
    alt.Y("count():Q", title="Count")
).properties(
    title="Drawdown area",
    width=400,
    height=300
)

histograms = alt.hconcat(*[chart_days, chart_area])
histograms.display()

# total number of events
total_events = df_transitions.shape[0]
print("Total number of events:", total_events)
# Count the number of events with no fall date
count_2023_end = df_transitions[df_transitions['Percentage_Increase_at_Return'] > 100*return_threshold].shape[0]
print("Event with no return date:", count_2023_end)
df_transitions[df_transitions.Years_to_return > 10]
# print(df_transitions[df_transitions.Area_Under_Curve > 300])
# df_transitions.describe()
event_counts = df_transitions.groupby('Country').size().reset_index(name='Event_Count')
event_counts

In [ ]:
# plot the normalised exchange rates with the devaluation events and export to PDF
import math
keep_from_extended = ["times", "values_integrand"]
final_charts = []
three_countries = ['AGO']
for i, country in enumerate(african_countries): #african_countries

    # Define the column names for the current country
    r2ner_col_norm = f"{country}_R2NER_normalized"
    rer_col_norm   = f"{country}_RER_normalized"
    ner_col_norm   = f"{country}_NER_normalized"
    ner_col   = f"{country}_NER"
    ma_col = f"{country}_R2NER_MA"

    legend_setting = alt.Legend(title="Exchange rates", 
                                orient= "bottom", 
                                labelExpr="substring(datum.label, 4, length(datum.label)) === 'NER' ? 'Nominal (right axis)' : datum.label === 'ma' ? 'Moving average' : 'RealLCY to NominalUSD '") if i%2 == 0 else None
    
    # Ensure the necessary columns exist in fX for this country.
    if all(col in fX.columns for col in [r2ner_col_norm, ner_col]):
        # Prepare data in long format
        data = fX[fX.index >= start_date].reset_index()[['index', ner_col, r2ner_col_norm, ma_col]]
        data_long = data.melt(
            id_vars=['index'],
            value_vars=[ner_col, r2ner_col_norm,ma_col],
            var_name='Metric',
            value_name='Normalized_Value'
        )
        
        # Prepare background intervals from transitions for the current country.
        trans = df_transitions[df_transitions['Country'] == country]
        end_date = fX.index.max()
        red_backgrounds = []
        blue_annotations = []
        j=-1
        for _, row in trans.iterrows():
            j=j+1
            # Shade the area below the R2NER curve between Trough_Date and Peak_Date
            event_extended = results_extended[country][j]
            extended_series = {k: event_extended[k] for k in keep_from_extended}
            data_extended = pd.DataFrame(extended_series)
            red_shade_data = data_extended[(data_extended["times"] >= row["Trough_Date"]) &
                                           (data_extended["times"] <= row["Fall_Date"])].assign(MA_at_Trough=row['MA_at_Trough'])
                                            # data[(data['index'] >= row['Trough_Date']) &
                                            #                   (data['index'] <= row['Peak_Date'])].assign(MA_at_Trough=row['MA_at_Trough'])
            red_rect = alt.Chart(red_shade_data).mark_area(
                opacity=0.2,
                color='red'
            ).encode(
                x=alt.X("times:T"),
                y=alt.Y("values_integrand:Q"),
                y2=alt.Y2("MA_at_Trough:Q")
            )
            red_backgrounds.append(red_rect)

            # Add blue text annotation under the trough_value with the value of Area_Under_Curve
            if pd.notnull(row['Area_Under_Curve']):
                x_pos_blue = row['Trough_Date'] + (row['Fall_Date'] - row['Trough_Date']) * 4/7
                blue_text_df = pd.DataFrame({'x': [x_pos_blue], 'y': [row['Trough_Value']+3], 'text': [f"{row['Area_Under_Curve']:.0f}%"]})
                blue_text = alt.Chart(blue_text_df).mark_text(fontSize=12, dy=5, color='red').encode(
                    x=alt.X('x:T'),
                    y=alt.Y('y:Q'),
                    text=alt.Text('text:N')
                )
                blue_annotations.append(blue_text)

        # Create two charts: one for metrics other than ner_col_norm (left y-axis) 
        # and one exclusively for ner_col_norm (right y-axis)

        left_data = data_long[data_long['Metric'] != ner_col]
        right_data = data_long[data_long['Metric'] == ner_col]
        
        left_chart = alt.layer(
            alt.Chart(left_data).mark_line(point=True).encode(
            x=alt.X("index:T", title="Year", axis=alt.Axis(format="%Y")),
            y=alt.Y("Normalized_Value:Q", title="Left Axis Value"),
            color=alt.Color("Metric:N"),
            tooltip=["index:T", "Metric", "Normalized_Value"]
            ),
            *red_backgrounds,
            *blue_annotations
        )
        
        right_chart = alt.Chart(right_data).mark_line(point=True).encode(
            x=alt.X("index:T", axis=alt.Axis(format="%Y")),
            y=alt.Y("Normalized_Value:Q", title="Right Axis Value", axis=alt.Axis(orient='right')),
            color=alt.Color("Metric:N", legend=legend_setting),
            tooltip=["index:T", "Metric", "Normalized_Value"]
        )
        
        # Layer the two charts and set independent y-scales for dual axes.
        base = alt.layer(left_chart, right_chart).resolve_scale(
            y='independent'
        )
    
        
        # Combine background intervals, annotations, and base chart.
        # Disable axes for the annotation layers to prevent duplicate y-axis labels.
        final_chart = alt.layer(base).properties(
            title=f"Real depreciation events in {iso3_to_country_name[country]}",
            width=400,
            height=400
        )
        # final_chart.display()
        # print(df_transitions[df_transitions.Country == country][['Country', 'Area_Under_Curve']])

        final_charts.append(final_chart)

# final_charts is assumed to be a list where each iteration appended the generated final_chart.
# For example, modify the loop in cell 2 to include:
#     final_charts.append(final_chart)
#
# This snippet arranges the charts into rows of 2 and exports them as a single PDF.
rows = math.ceil(len(final_charts) / 2)
chart_rows = []
for i in range(rows):
    charts_in_row = final_charts[i*2 : i*2 + 2]
    if len(charts_in_row) == 2:
        # Resolve the color scales independently so that each chart retains its own color encoding.
        row = alt.hconcat(*charts_in_row).resolve_scale(color='independent')
    else:
        row = charts_in_row[0]
    chart_rows.append(row)

final_layout = alt.vconcat(*chart_rows)
# final_layout.save("chartstest.pdf")
final_layout

In [ ]:
# j, results_extended[country][1]
country

In [ ]:
results_extended[country][j]["values_extended"]

In [ ]:
import altair as alt
from vega_datasets import data

# Prepare the TopoJSON and your data for joining
countries = alt.topo_feature(data.world_110m.url, 'countries')

# Prepare a DataFrame with ISO3 and Area_Under_Curve (or any variable)
# Create a base DataFrame with all African countries and a default area_under_curve value of 0
base_df = pd.DataFrame({'Country': african_countries, 'Area_Under_Curve': 0, 'Years_to_return': 0})

# From df_transitions, select only the transitions for African countries and, for each country, get the maximum Area_Under_Curve
transitions_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby('Country', as_index=False)['Area_Under_Curve']
    .max()
)
length_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby("Country", as_index=False)['Years_to_return'].max()
)

transitions_max = pd.merge(transitions_max, length_max, on='Country', how='outer')

# Merge the base DataFrame with the maximum transition values; countries without transitions remain 0
africa_choropleth = pd.merge(base_df, transitions_max, on='Country', how='left', suffixes=('', '_trans'))
africa_choropleth['Area_Under_Curve'] = africa_choropleth['Area_Under_Curve_trans'].fillna(africa_choropleth['Area_Under_Curve'])
# africa_choropleth = africa_choropleth[['Country', 'Area_Under_Curve', 'Years_to_return']]

# Create a mapping from ISO 3166-1 alpha-3 to ISO 3166-1 numeric codes
iso3166_alpha3_to_num = {
    'DZA': 12, 'AGO': 24, 'BEN': 204, 'BWA': 72, 'BFA': 854, 'BDI': 108, 'CMR': 120, 'CPV': 132,
    'CAF': 140, 'TCD': 148, 'COM': 174, 'COG': 178, 'CIV': 384, 'COD': 180, 'DJI': 262, 'EGY': 818,
    'GNQ': 226, 'ERI': 232, 'SWZ': 748, 'ETH': 231, 'GAB': 266, 'GMB': 270, 'GHA': 288, 'GIN': 324,
    'GNB': 624, 'KEN': 404, 'LSO': 426, 'LBR': 430, 'MDG': 450, 'MWI': 454, 'MLI': 466, 'MRT': 478,
    'MUS': 480, 'MAR': 504, 'MOZ': 508, 'NAM': 516, 'NER': 562, 'NGA': 566, 'RWA': 646, 'STP': 678,
    'SEN': 686, 'SYC': 690, 'SLE': 694, 'SOM': 706, 'ZAF': 710, 'SSD': 728, 'SDN': 729, 'TZA': 834,
    'TGO': 768, 'TUN': 788, 'UGA': 800, 'ZMB': 894, 'ZWE': 716, 'LBY': 434	
}

# Convert the iso3166_alpha3_to_num mapping to a DataFrame and merge with africa_choropleth
iso_mapping_df = pd.DataFrame(list(iso3166_alpha3_to_num.items()), columns=['Country', 'ISO_3166_1_num'])
africa_choropleth = africa_choropleth.merge(iso_mapping_df, on='Country', how='left')

# Join the full country names using the iso3_to_country_name dictionary
africa_choropleth['Country_Name'] = africa_choropleth['Country'].map(iso3_to_country_name)

# List of numeric codes for filtering (used here only for lookup)
africa_numeric_ids = list(iso3166_alpha3_to_num.values())

# Create the map: all countries are plotted, and those missing Area_Under_Curve are filled white.
choropleth = alt.Chart(countries).mark_geoshape(
    stroke='green'  # add a border to separate countries from the sea
).encode(
    # If a country has a value then show the color, otherwise fill white.
    color=alt.condition(
        "datum.Area_Under_Curve != null",
        alt.Color('Area_Under_Curve:Q', scale=alt.Scale(scheme='blues'), title='Area Under Curve'),
        alt.value("white")
    ),
    tooltip=[
        alt.Tooltip('Country_Name:N', title='Country'),
        alt.Tooltip('Area_Under_Curve:Q', title='Area Under Curve')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(africa_choropleth, 'ISO_3166_1_num', ['Area_Under_Curve', 'Country_Name'])
).project(
    type="mercator",
    center=[17, 0],   # Center on Africa
    scale=250
).properties(
    width=500,
    height=500,
    title='Choropleth of Area Under Curve in Africa'
)

choropleth.display()

In [ ]:
countries = alt.topo_feature(data.world_110m.url, 'countries')

# Prepare a DataFrame with ISO3 and Area_Under_Curve (or any variable)
# Create a base DataFrame with all African countries and a default area_under_curve value of 0
base_df = pd.DataFrame({'Country': african_countries, 'Area_Under_Curve': 0, 'Years_to_return': 0})

# From df_transitions, select only the transitions for African countries and, for each country, get the maximum Area_Under_Curve
transitions_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby('Country', as_index=False)['Area_Under_Curve']
    .max()
)
length_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby("Country", as_index=False)['Years_to_return'].max()
)

transitions_max = pd.merge(transitions_max, length_max, on='Country', how='outer')

# Merge the base DataFrame with the maximum transition values; countries without transitions remain 0
africa_choropleth = pd.merge(base_df, transitions_max, on='Country', how='left', suffixes=('', '_trans'))
africa_choropleth['Area_Under_Curve'] = africa_choropleth['Area_Under_Curve_trans'].fillna(africa_choropleth['Area_Under_Curve'])
# africa_choropleth = africa_choropleth[['Country', 'Area_Under_Curve', 'Years_to_return']]

In [ ]:
africa_choropleth = pd.merge(base_df, transitions_max, on='Country', how='left', suffixes=('', '_trans'))
africa_choropleth